# RAG Exploration - OeNB Daten

Dieses Notebook führt durch:
1. **Datenqualität prüfen** - Statistiken, leere Seiten, Duplikate, HTML-Reste
2. **RAG Setup** - Ollama, ChromaDB, Chunking
3. **RAG Testen** - Fragen stellen, Quellen anzeigen

---

## 1. Setup & Laden

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# SQLite verbinden
DB_PATH = Path("../data/pages.db")
conn = sqlite3.connect(DB_PATH)

# Basis-Info
pages = pd.read_sql("SELECT COUNT(*) as count FROM pages", conn).iloc[0, 0]
bodies = pd.read_sql("SELECT COUNT(*) as count FROM page_bodies", conn).iloc[0, 0]

# Check ob page_content existiert
try:
    content = pd.read_sql("SELECT COUNT(*) as count FROM page_content", conn).iloc[0, 0]
except:
    content = 0

print(f"Pages:   {pages:,}")
print(f"Bodies:  {bodies:,}")
print(f"Content: {content:,}" + (" (Text-Extraktion noch nicht ausgeführt!)" if content == 0 else ""))

## 2. Datenqualität prüfen

### 2.1 Statistik-Übersicht

In [ ]:
# Statistik-Übersicht
if content > 0:
    stats = pd.read_sql('''
        SELECT
            COUNT(*) as total_pages,
            ROUND(AVG(LENGTH(text_content)), 0) as avg_text_len,
            MIN(LENGTH(text_content)) as min_text_len,
            MAX(LENGTH(text_content)) as max_text_len
        FROM page_content
    ''', conn)
    print("=== Statistiken ===")
    print(f"Seiten:          {stats['total_pages'].iloc[0]:,}")
    print(f"Ø Textlänge:     {stats['avg_text_len'].iloc[0]:,.0f} Zeichen")
    print(f"Min Textlänge:   {stats['min_text_len'].iloc[0]:,} Zeichen")
    print(f"Max Textlänge:   {stats['max_text_len'].iloc[0]:,} Zeichen")
else:
    print("⚠️ Erst Text-Extraktion ausführen: python analysis/extract_text.py data/pages.db")

### 2.2 Leere / kurze Seiten

In [ ]:
# Leere oder sehr kurze Seiten
if content > 0:
    short_pages = pd.read_sql('''
        SELECT p.url, pc.title, LENGTH(pc.text_content) as text_len
        FROM page_content pc
        JOIN pages p ON p.id = pc.page_id
        WHERE LENGTH(pc.text_content) < 100
        ORDER BY text_len
        LIMIT 20
    ''', conn)
    print(f"=== Kurze Seiten (<100 Zeichen): {len(short_pages)} ===")
    if len(short_pages) > 0:
        display(short_pages)
    else:
        print("Keine kurzen Seiten gefunden!")

### 2.3 Duplikate

In [ ]:
# Duplikate (gleicher Text)
if content > 0:
    duplicates = pd.read_sql('''
        SELECT SUBSTR(text_content, 1, 100) as text_preview, COUNT(*) as count
        FROM page_content
        WHERE LENGTH(text_content) > 50
        GROUP BY text_content
        HAVING COUNT(*) > 1
        ORDER BY count DESC
        LIMIT 10
    ''', conn)
    print(f"=== Duplikate: {len(duplicates)} verschiedene Texte mehrfach ===")
    if len(duplicates) > 0:
        display(duplicates)
    else:
        print("Keine Duplikate gefunden!")

### 2.4 HTML-Reste im Text

In [ ]:
# HTML-Reste im Text
import re

if content > 0:
    html_pattern = re.compile(r'<[a-zA-Z][^>]*>')
    
    sample = pd.read_sql('''
        SELECT page_id, title, text_content
        FROM page_content
        WHERE LENGTH(text_content) > 0
    ''', conn)
    
    sample['has_html'] = sample['text_content'].apply(
        lambda x: bool(html_pattern.search(str(x))) if x else False
    )
    issues = sample[sample['has_html']]
    
    pct = len(issues) / len(sample) * 100 if len(sample) > 0 else 0
    print(f"=== HTML-Reste: {len(issues)} von {len(sample)} ({pct:.1f}%) ===")
    
    if len(issues) > 0:
        print("\nBeispiele:")
        for _, row in issues.head(5).iterrows():
            matches = html_pattern.findall(str(row['text_content']))[:3]
            print(f"  - {row['title'][:50]}: {matches}")

### 2.5 Sprachen-Verteilung

In [ ]:
# Sprachen-Verteilung
if content > 0:
    languages = pd.read_sql('''
        SELECT language, COUNT(*) as count
        FROM page_content
        GROUP BY language
        ORDER BY count DESC
    ''', conn)
    print("=== Sprachen ===")
    for _, row in languages.iterrows():
        print(f"  {row['language']}: {row['count']:,}")

### 2.6 Section-Abdeckung

In [ ]:
# Section-Abdeckung
if content > 0:
    sections = pd.read_sql('''
        SELECT page_section, COUNT(*) as count
        FROM page_content
        GROUP BY page_section
        ORDER BY count DESC
        LIMIT 15
    ''', conn)
    print("=== Top 15 Sections ===")
    for _, row in sections.iterrows():
        print(f"  {row['page_section']}: {row['count']:,}")

### 2.7 Stichproben

In [ ]:
# Stichproben (10 zufällige Seiten)
if content > 0:
    samples = pd.read_sql('''
        SELECT p.url, pc.title, SUBSTR(pc.text_content, 1, 300) as preview
        FROM page_content pc
        JOIN pages p ON p.id = pc.page_id
        WHERE LENGTH(pc.text_content) > 100
        ORDER BY RANDOM()
        LIMIT 10
    ''', conn)
    
    print("=== 10 Zufällige Stichproben ===\n")
    for i, row in samples.iterrows():
        print(f"--- {row['title'][:60]} ---")
        print(f"URL: {row['url']}")
        print(f"Text: {row['preview']}...")
        print()

## 3. Text-Extraktion (falls nötig)

Falls `page_content` noch leer ist, führe diesen Befehl im Terminal aus:

```bash
source venv/bin/activate
python analysis/extract_text.py data/pages.db
```

Dann Notebook neu starten und Abschnitt 1 erneut ausführen.

## 4. RAG Setup

### 4.1 Ollama prüfen

Stelle sicher, dass Ollama läuft und die Modelle installiert sind:

```bash
ollama list
# Sollte zeigen: nomic-embed-text, gemma:2b

# Falls nicht:
ollama pull nomic-embed-text
ollama pull gemma:2b
```

In [ ]:
# Packages importieren
try:
    from langchain_ollama import OllamaEmbeddings, OllamaLLM
    from langchain_chroma import Chroma
    print("✅ LangChain + Ollama geladen")
except ImportError as e:
    print(f"❌ Fehler: {e}")
    print("\nInstalliere mit:")
    print("pip install langchain langchain-ollama langchain-chroma chromadb")

In [ ]:
# Ollama Modelle laden
try:
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    llm = OllamaLLM(model="gemma:2b")  # Kleines Modell für CPU
    
    # Test
    test_embed = embeddings.embed_query("Test")
    print(f"✅ Embedding Modell geladen (Dimension: {len(test_embed)})")
    print("✅ LLM geladen")
except Exception as e:
    print(f"❌ Fehler: {e}")
    print("\nStelle sicher dass Ollama läuft: ollama serve")

### 4.2 Daten laden und chunken

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Daten laden
df = pd.read_sql('''
    SELECT p.url, pc.title, pc.text_content, pc.page_section
    FROM page_content pc
    JOIN pages p ON p.id = pc.page_id
    WHERE LENGTH(pc.text_content) > 100
''', conn)

print(f"Geladene Dokumente: {len(df):,}")

In [ ]:
# Text Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

documents = []
for _, row in df.iterrows():
    if not row['text_content']:
        continue
    chunks = text_splitter.split_text(row['text_content'])
    for chunk in chunks:
        documents.append(Document(
            page_content=chunk,
            metadata={
                "url": row['url'],
                "title": row['title'] or "Kein Titel",
                "section": row['page_section'] or "Unbekannt"
            }
        ))

print(f"Chunks erstellt: {len(documents):,}")
print(f"Ø Chunks pro Seite: {len(documents)/len(df):.1f}")

### 4.3 Vector Store erstellen

⚠️ **Achtung:** Das dauert bei ~27.000 Seiten einige Minuten!

In [ ]:
import os
import shutil

CHROMA_PATH = "../data/chromadb"

# Löschen falls existiert (für Neuaufbau)
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print("Alter Vector Store gelöscht")

print(f"Erstelle Vector Store mit {len(documents):,} Chunks...")
print("(Das kann einige Minuten dauern...)")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)

print(f"\n✅ Vector Store erstellt und gespeichert in {CHROMA_PATH}")

## 5. RAG Testen

In [ ]:
from langchain.chains import RetrievalQA

# RAG Chain erstellen
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True
)

print("✅ RAG Chain erstellt")

In [ ]:
def ask(question: str):
    """Frage stellen und Antwort mit Quellen anzeigen."""
    print("=" * 70)
    print(f"FRAGE: {question}")
    print("=" * 70)
    
    result = qa_chain.invoke({"query": question})
    
    print(f"\nANTWORT:\n{result['result']}")
    print("\n" + "-" * 70)
    print("QUELLEN:")
    for i, doc in enumerate(result['source_documents'], 1):
        title = doc.metadata.get('title', 'Kein Titel')[:50]
        url = doc.metadata.get('url', '')
        section = doc.metadata.get('section', '')
        text = doc.page_content[:100]
        print(f"\n[{i}] {title}...")
        print(f"    URL: {url}")
        print(f"    Section: {section}")
        print(f"    Text: {text}...")
    print("=" * 70)

### Test-Fragen

In [ ]:
ask("Was ist der aktuelle Leitzins der EZB?")

In [ ]:
ask("Wie funktioniert die Zahlungsbilanz?")

In [ ]:
ask("Welche Statistiken gibt es zu Wohnbaukrediten?")

In [ ]:
ask("Was sind standardisierte Tabellen?")

### Eigene Fragen

In [ ]:
# Eigene Frage hier eingeben:
ask("DEINE FRAGE HIER")

## 6. Erkenntnisse & Nächste Schritte

### Was funktioniert gut?
- 

### Was muss verbessert werden?
- 

### Nächste Schritte:
- 

### Notizen:
- 

---
## Cleanup (optional)

Vector Store löschen um Speicherplatz freizugeben:

In [ ]:
# Uncomment to delete:
# import shutil
# shutil.rmtree("../data/chromadb")
# print("Vector Store gelöscht")